In [1]:
!pip install --quiet peft bitsandbytes accelerate transformers datasets
!pip install --quiet git+https://github.com/huggingface/transformers.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 23.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.7 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 

In [2]:
from huggingface_hub import login
login(token="hf_token")

In [116]:
import os
import torch
import pandas as pd
from PIL import Image
from transformers import BlipProcessor, BlipForQuestionAnswering, TrainingArguments, Trainer
from torch.utils.data import Dataset, DataLoader
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig
from sklearn.metrics import accuracy_score, f1_score

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [23]:
df = pd.read_csv("/kaggle/input/vqa-dataset/Subset/combined_vqa_single_answer.csv")
base_img_dir = "/kaggle/input/vqa-dataset/Subset/images"
df["full_path"] = df["image_path"].apply(lambda x: os.path.join(base_img_dir, x))

In [59]:
from transformers import BlipForQuestionAnswering

class CustomBlipForQuestionAnswering(BlipForQuestionAnswering):
    def forward(self, 
                input_ids=None, 
                pixel_values=None, 
                attention_mask=None, 
                labels=None, 
                **kwargs):
        if 'inputs_embeds' in kwargs:
            kwargs.pop('inputs_embeds')

        return super().forward(
            input_ids=input_ids,
            pixel_values=pixel_values,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs
        )

In [82]:
from transformers.models.blip.modeling_blip_text import BlipTextEmbeddings

orig_forward = BlipTextEmbeddings.forward

def patched_forward(self, input_ids=None, position_ids=None, inputs_embeds=None, past_key_values_length=0):
    if input_ids is not None:
        input_shape = input_ids.size()
    elif inputs_embeds is not None:
        input_shape = inputs_embeds.size()[:-1]
    else:
        raise ValueError("You have to specify either input_ids or inputs_embeds")

    if position_ids is None:
        position_ids = torch.arange(
            past_key_values_length, input_shape[-1] + past_key_values_length, dtype=torch.long, device=self.position_ids.device
        ).unsqueeze(0).expand(input_shape)

    if inputs_embeds is None:
        inputs_embeds = self.word_embeddings(input_ids)

    position_embeddings = self.position_embeddings(position_ids)
    
    embeddings = inputs_embeds + position_embeddings
    embeddings = self.LayerNorm(embeddings)
    embeddings = self.dropout(embeddings)
    return embeddings

BlipTextEmbeddings.forward = patched_forward

In [83]:
processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
bnb_config = BitsAndBytesConfig( load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16 )
model = CustomBlipForQuestionAnswering.from_pretrained( "Salesforce/blip-vqa-base", quantization_config=bnb_config)
model = prepare_model_for_kbit_training(model)
model.to(device)

CustomBlipForQuestionAnswering(
  (vision_model): BlipVisionModel(
    (embeddings): BlipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): BlipEncoder(
      (layers): ModuleList(
        (0-11): 12 x BlipEncoderLayer(
          (self_attn): BlipAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (qkv): Linear4bit(in_features=768, out_features=2304, bias=True)
            (projection): Linear4bit(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): BlipMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear4bit(in_features=768, out_features=3072, bias=True)
            (fc2): Linear4bit(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNor

In [84]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 228,959,036
Trainable parameters: 0


In [19]:
#for name, module in model.named_modules():
#    print(name)

In [85]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=[
        "text_encoder.encoder.layer.0.attention.self.query",
        "text_encoder.encoder.layer.0.attention.self.key",
        "text_encoder.encoder.layer.0.attention.self.value",
        "text_encoder.encoder.layer.0.crossattention.self.query",
        "text_encoder.encoder.layer.0.crossattention.self.key",
        "text_encoder.encoder.layer.0.crossattention.self.value",
        "text_encoder.encoder.layer.1.attention.self.query",
        "text_encoder.encoder.layer.1.attention.self.key",
        "text_encoder.encoder.layer.1.attention.self.value",
        "text_encoder.encoder.layer.1.crossattention.self.query",
        "text_encoder.encoder.layer.1.crossattention.self.key",
        "text_encoder.encoder.layer.1.crossattention.self.value",
        "text_encoder.encoder.layer.2.attention.self.query",
        "text_encoder.encoder.layer.2.attention.self.key",
        "text_encoder.encoder.layer.2.attention.self.value",
        "text_encoder.encoder.layer.2.crossattention.self.query",
        "text_encoder.encoder.layer.2.crossattention.self.key",
        "text_encoder.encoder.layer.2.crossattention.self.value",
        "text_encoder.encoder.layer.3.attention.self.query",
        "text_encoder.encoder.layer.3.attention.self.key",
        "text_encoder.encoder.layer.3.attention.self.value",
        "text_encoder.encoder.layer.3.crossattention.self.query",
        "text_encoder.encoder.layer.3.crossattention.self.key",
        "text_encoder.encoder.layer.3.crossattention.self.value",
        "text_encoder.encoder.layer.4.attention.self.query",
        "text_encoder.encoder.layer.4.attention.self.key",
        "text_encoder.encoder.layer.4.attention.self.value",
        "text_encoder.encoder.layer.4.crossattention.self.query",
        "text_encoder.encoder.layer.4.crossattention.self.key",
        "text_encoder.encoder.layer.4.crossattention.self.value",
        "text_encoder.encoder.layer.5.attention.self.query",
        "text_encoder.encoder.layer.5.attention.self.key",
        "text_encoder.encoder.layer.5.attention.self.value",
        "text_encoder.encoder.layer.5.crossattention.self.query",
        "text_encoder.encoder.layer.5.crossattention.self.key",
        "text_encoder.encoder.layer.5.crossattention.self.value",
        "text_encoder.encoder.layer.6.attention.self.query",
        "text_encoder.encoder.layer.6.attention.self.key",
        "text_encoder.encoder.layer.6.attention.self.value",
        "text_encoder.encoder.layer.6.crossattention.self.query",
        "text_encoder.encoder.layer.6.crossattention.self.key",
        "text_encoder.encoder.layer.6.crossattention.self.value",
        "text_encoder.encoder.layer.7.attention.self.query",
        "text_encoder.encoder.layer.7.attention.self.key",
        "text_encoder.encoder.layer.7.attention.self.value",
        "text_encoder.encoder.layer.7.crossattention.self.query",
        "text_encoder.encoder.layer.7.crossattention.self.key",
        "text_encoder.encoder.layer.7.crossattention.self.value",
        "text_encoder.encoder.layer.8.attention.self.query",
        "text_encoder.encoder.layer.8.attention.self.key",
        "text_encoder.encoder.layer.8.attention.self.value",
        "text_encoder.encoder.layer.8.crossattention.self.query",
        "text_encoder.encoder.layer.8.crossattention.self.key",
        "text_encoder.encoder.layer.8.crossattention.self.value",
        "text_encoder.encoder.layer.9.attention.self.query",
        "text_encoder.encoder.layer.9.attention.self.key",
        "text_encoder.encoder.layer.9.attention.self.value",
        "text_encoder.encoder.layer.9.crossattention.self.query",
        "text_encoder.encoder.layer.9.crossattention.self.key",
        "text_encoder.encoder.layer.9.crossattention.self.value",
        "text_encoder.encoder.layer.10.attention.self.query",
        "text_encoder.encoder.layer.10.attention.self.key",
        "text_encoder.encoder.layer.10.attention.self.value",
        "text_encoder.encoder.layer.10.crossattention.self.query",
        "text_encoder.encoder.layer.10.crossattention.self.key",
        "text_encoder.encoder.layer.10.crossattention.self.value",
        "text_encoder.encoder.layer.11.attention.self.query",
        "text_encoder.encoder.layer.11.attention.self.key",
        "text_encoder.encoder.layer.11.attention.self.value",
        "text_encoder.encoder.layer.11.crossattention.self.query",
        "text_encoder.encoder.layer.11.crossattention.self.key",
        "text_encoder.encoder.layer.11.crossattention.self.value",
        "text_decoder.bert.encoder.layer.0.attention.self.query",
        "text_decoder.bert.encoder.layer.0.attention.self.key",
        "text_decoder.bert.encoder.layer.0.attention.self.value",
        "text_decoder.bert.encoder.layer.0.crossattention.self.query",
        "text_decoder.bert.encoder.layer.0.crossattention.self.key",
        "text_decoder.bert.encoder.layer.0.crossattention.self.value",
        "text_decoder.bert.encoder.layer.1.attention.self.query",
        "text_decoder.bert.encoder.layer.1.attention.self.key",
        "text_decoder.bert.encoder.layer.1.attention.self.value",
        "text_decoder.bert.encoder.layer.1.crossattention.self.query",
        "text_decoder.bert.encoder.layer.1.crossattention.self.key",
        "text_decoder.bert.encoder.layer.1.crossattention.self.value",
        "text_decoder.bert.encoder.layer.2.attention.self.query",
        "text_decoder.bert.encoder.layer.2.attention.self.key",
        "text_decoder.bert.encoder.layer.2.attention.self.value",
        "text_decoder.bert.encoder.layer.2.crossattention.self.query",
        "text_decoder.bert.encoder.layer.2.crossattention.self.key",
        "text_decoder.bert.encoder.layer.2.crossattention.self.value",
        "text_decoder.bert.encoder.layer.3.attention.self.query",
        "text_decoder.bert.encoder.layer.3.attention.self.key",
        "text_decoder.bert.encoder.layer.3.attention.self.value",
        "text_decoder.bert.encoder.layer.3.crossattention.self.query",
        "text_decoder.bert.encoder.layer.3.crossattention.self.key",
        "text_decoder.bert.encoder.layer.3.crossattention.self.value",
        "text_decoder.bert.encoder.layer.4.attention.self.query",
        "text_decoder.bert.encoder.layer.4.attention.self.key",
        "text_decoder.bert.encoder.layer.4.attention.self.value",
        "text_decoder.bert.encoder.layer.4.crossattention.self.query",
        "text_decoder.bert.encoder.layer.4.crossattention.self.key",
        "text_decoder.bert.encoder.layer.4.crossattention.self.value",
        "text_decoder.bert.encoder.layer.5.attention.self.query",
        "text_decoder.bert.encoder.layer.5.attention.self.key",
        "text_decoder.bert.encoder.layer.5.attention.self.value",
        "text_decoder.bert.encoder.layer.5.crossattention.self.query",
        "text_decoder.bert.encoder.layer.5.crossattention.self.key",
        "text_decoder.bert.encoder.layer.5.crossattention.self.value",
        "text_decoder.bert.encoder.layer.6.attention.self.query",
        "text_decoder.bert.encoder.layer.6.attention.self.key",
        "text_decoder.bert.encoder.layer.6.attention.self.value",
        "text_decoder.bert.encoder.layer.6.crossattention.self.query",
        "text_decoder.bert.encoder.layer.6.crossattention.self.key",
        "text_decoder.bert.encoder.layer.6.crossattention.self.value",
        "text_decoder.bert.encoder.layer.7.attention.self.query",
        "text_decoder.bert.encoder.layer.7.attention.self.key",
        "text_decoder.bert.encoder.layer.7.attention.self.value",
        "text_decoder.bert.encoder.layer.7.crossattention.self.query",
        "text_decoder.bert.encoder.layer.7.crossattention.self.key",
        "text_decoder.bert.encoder.layer.7.crossattention.self.value",
        "text_decoder.bert.encoder.layer.8.attention.self.query",
        "text_decoder.bert.encoder.layer.8.attention.self.key",
        "text_decoder.bert.encoder.layer.8.attention.self.value",
        "text_decoder.bert.encoder.layer.8.crossattention.self.query",
        "text_decoder.bert.encoder.layer.8.crossattention.self.key",
        "text_decoder.bert.encoder.layer.8.crossattention.self.value",
        "text_decoder.bert.encoder.layer.9.attention.self.query",
        "text_decoder.bert.encoder.layer.9.attention.self.key",
        "text_decoder.bert.encoder.layer.9.attention.self.value",
        "text_decoder.bert.encoder.layer.9.crossattention.self.query",
        "text_decoder.bert.encoder.layer.9.crossattention.self.key",
        "text_decoder.bert.encoder.layer.9.crossattention.self.value",
        "text_decoder.bert.encoder.layer.10.attention.self.query",
        "text_decoder.bert.encoder.layer.10.attention.self.key",
        "text_decoder.bert.encoder.layer.10.attention.self.value",
        "text_decoder.bert.encoder.layer.10.crossattention.self.query",
        "text_decoder.bert.encoder.layer.10.crossattention.self.key",
        "text_decoder.bert.encoder.layer.10.crossattention.self.value",
        "text_decoder.bert.encoder.layer.11.attention.self.query",
        "text_decoder.bert.encoder.layer.11.attention.self.key",
        "text_decoder.bert.encoder.layer.11.attention.self.value",
        "text_decoder.bert.encoder.layer.11.crossattention.self.query",
        "text_decoder.bert.encoder.layer.11.crossattention.self.key",
        "text_decoder.bert.encoder.layer.11.crossattention.self.value",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_config)

In [86]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Added LoRA Trainable parameters: {trainable_params:,}")

Total parameters: 230,728,508
Added LoRA Trainable parameters: 1,769,472


In [87]:
class VQADataset(Dataset):
    def __init__(self, dataframe, processor):
        self.dataset = dataframe
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        question = self.dataset.iloc[idx]['question']
        answer = self.dataset.iloc[idx]['answer']
        answer = str(answer)
        image_path = self.dataset.iloc[idx]['full_path']
        image = Image.open(image_path).convert("RGB")
        text = question
        
        encoding = self.processor(image, text, padding="max_length", truncation=True, return_tensors="pt")
        labels = self.processor.tokenizer(
            answer, max_length= 8, padding="max_length", truncation=True, return_tensors='pt'
        ).input_ids.squeeze()
        encoding["labels"] = labels
        for k,v in encoding.items():  encoding[k] = v.squeeze()
        return encoding


train_dataset = VQADataset(df, processor)
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=False, pin_memory=True)

In [88]:
repo_name = "blip-vqa-qlora"
from tqdm import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9, last_epoch=-1, verbose=False)
patience = 10
min_eval_loss = float("inf")
early_stopping_hook = 0
tracking_information = []
scaler = torch.cuda.amp.GradScaler()
model.train()

for epoch in range(3):
    print(f"Epoch {epoch + 1}")
    total_loss = 0
    for batch in tqdm(train_dataloader):
        input_ids = batch.pop('input_ids').to(device)
        pixel_values = batch.pop('pixel_values').to(device)
        attention_masked = batch.pop('attention_mask').to(device)
        labels = batch.pop('labels').to(device)

        if 'inputs_embeds' in batch:
            batch.pop('inputs_embeds')
        
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(input_ids=input_ids,
                        pixel_values=pixel_values,
                        attention_mask=attention_masked,
                        labels=labels)

        loss = outputs.loss
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
    print(f"Average loss: {total_loss / len(train_dataloader):.4f}")


/tmp/ipykernel_35/3088090489.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1


100%|██████████| 2111/2111 [26:43<00:00,  1.32it/s]


Average loss: 7.3937
Epoch 2


100%|██████████| 2111/2111 [26:46<00:00,  1.31it/s]


Average loss: 6.4027
Epoch 3


100%|██████████| 2111/2111 [26:44<00:00,  1.32it/s]

Average loss: 6.3287


In [89]:
processor.push_to_hub(f"Soumik1996/{repo_name}")
model.push_to_hub(f"Soumik1996/{repo_name}")

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/7.12M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Soumik1996/blip-vqa-qlora/commit/53ba65b21b33fcf0dfa2aef939c3b4a99758bf6f', commit_message='Upload model', commit_description='', oid='53ba65b21b33fcf0dfa2aef939c3b4a99758bf6f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Soumik1996/blip-vqa-qlora', endpoint='https://huggingface.co', repo_type='model', repo_id='Soumik1996/blip-vqa-qlora'), pr_revision=None, pr_num=None)

In [111]:
from transformers import AutoProcessor, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
import torch

peft_config = PeftConfig.from_pretrained("Soumik1996/blip-vqa-qlora")
base_model_name = peft_config.base_model_name_or_path

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained("Soumik1996/blip-vqa-qlora")
# print(base_model_name)
base_model = CustomBlipForQuestionAnswering.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, "Soumik1996/blip-vqa-qlora")

In [112]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 230,728,508
Trainable parameters: 0


In [113]:
df = pd.read_csv("/kaggle/input/vqa-inference-dataset/Inference/combined_inference_vqa_single_answer.csv")
df.head()

,image_path,question,answer
0,64/6412af43.jpg,What is the visible color of the chair?,Gray
1,64/6412af43.jpg,What is the average height of the back legs in...,4.125
2,64/64acb3aa.jpg,What kind of food is in the package?,Tortillas
3,64/64acb3aa.jpg,How many tortillas in total can the packaging ...,Six
4,64/64c3be6d.jpg,What is the display called?,Monitor


In [114]:
base_img_dir = "/kaggle/input/vqa-inference-dataset/Inference/images"
predictions = []
ground_truths = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    rel_path = row["image_path"]
    full_img_path = os.path.join(base_img_dir, rel_path)
    question = row["question"]
    true_answer = str(row["answer"]).strip().lower()

    try:
        image = Image.open(full_img_path).convert("RGB")
    except Exception as e:
        print(f"Failed to load image at {image_path}: {e}")
        continue

    inputs = processor(image, question, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=1)
    pred_answer = processor.decode(out[0], skip_special_tokens=True).strip().lower()

    predictions.append(pred_answer)
    ground_truths.append(true_answer)

100%|██████████| 2969/2969 [06:42<00:00,  7.38it/s]


In [117]:
# Exact string match accuracy
acc = accuracy_score(ground_truths, predictions)
f1 = f1_score(ground_truths, predictions, average='macro')

print(f"Accuracy: {acc:.4f}")
print(f"F1Score (macro): {f1:.4f}")

Accuracy: 0.4385
F1Score (macro): 0.1027


In [118]:
# Other evaluation metrics
!pip install evaluate bert-score rouge-score --quiet

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.5 MB/s eta 0:00:00


In [119]:
import evaluate
from tqdm import tqdm

bert_score = evaluate.load("bertscore")
rouge = evaluate.load("rouge")

In [120]:
rouge_results = rouge.compute(predictions=predictions, references=ground_truths)
print(rouge_results)

{'rouge1': 0.4416750870102167, 'rouge2': 0.0, 'rougeL': 0.4414505445155495, 'rougeLsum': 0.4417312226338834}


In [121]:
bert_results = bert_score.compute(predictions=predictions, references=ground_truths, lang="en")
print(f"BERTScore Precision: {sum(bert_results['precision'])/len(predictions):.4f}")
print(f"BERTScore Recall: {sum(bert_results['recall'])/len(predictions):.4f}")
print(f"BERTScore F1: {sum(bert_results['f1'])/len(predictions):.4f}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore Precision: 0.9766
BERTScore Recall: 0.9632
BERTScore F1: 0.9693


In [122]:
!git clone https://github.com/neulab/BARTScore.git

%cd BARTScore
!pip install -r requirements.txt --quiet
!pip install transformers==4.11.3 --quiet 

import sys
sys.path.append('/kaggle/working/BARTScore')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Cloning into 'BARTScore'...
remote: Enumerating objects: 220, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 220 (delta 18), reused 14 (delta 14), pack-reused 194 (from 1)
Receiving objects: 100% (220/220), 101.98 MiB | 24.21 MiB/s, done.
Resolving deltas: 100% (47/47), done.
Updating files: 100% (192/192), done.
/kaggle/working/BARTScore


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Could not find a version that satisfies the requirement BLEURT==0.0.2 (from versions: none)
ERROR: No matching distribution found for BLEURT==0.0.2


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 10.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 57.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 46.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [123]:
from bart_score import BARTScorer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
bart_scorer = BARTScorer(device=device, checkpoint='facebook/bart-large-cnn')
scores = bart_scorer.score(predictions, ground_truths, batch_size=8)
print(f"Avg BARTScore: {sum(scores)/len(scores):.4f}")

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Avg BARTScore: -4.6134
